# Grover's Algorithm

Grover’s algorithm is used to search for a marked item inside an unsorted search space.

What makes this algorithm interesting is that it increases the probability of the correct answer using interference and amplitude amplification instead of checking possibilities one-by-one

Suppose we have 4 possible states:

00
01
10
11

and only one of them is the correct answer, say 11

Classically, we would usually need to check states one-by-one until we eventually find the correct one.

Grover’s algorithm works differently.

Instead of checking states individually, it:
- creates a superposition of all states
- marks the correct state using phase
- uses interference to amplify the probability of the correct answer

# Why Grover Is Faster

Initially, all states have equal probability.

The oracle flips the phase of the correct state.

At first this does not change measurement probabilities directly.

The diffusion step causes interference:
- amplitudes of wrong answers decrease
- amplitude of the correct answer increases

Repeating this process gradually concentrates probability onto the marked state.

Initially:
▅ ▅ ▅ ▅

After one Grover iteration:
▃ ▃ ▃ █

After more iterations:
▁ ▁ ▁ ███

Grover’s algorithm requires roughly sqrt(N) operations instead of N for classical search.

The oracle already knows how to recognize the correct answer.

For example: |11> → -|11>
while all other states remain unchanged.

The algorithm itself does not initially know which state is correct.

It only uses the interference pattern created by the oracle to amplify the marked state.

E.g.: Your computer does not reveal the password directly, but it can still recognize whether a guess is correct.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# create circuit
qc = QuantumCircuit(2,2)

# create equal superposition
qc.h(0)
qc.h(1)

# oracle
# mark |11> by flipping its phase
qc.cz(0,1)

# diffusion operator

# apply hadamards
qc.h(0)
qc.h(1)

# apply X gates
# this temporarily maps |00> to |11>
qc.x(0)
qc.x(1)

# CZ flips phase only for |11>
# because of the X gates above,
# this effectively flips phase for |00>
qc.cz(0,1)

# undo the mapping
qc.x(0)
qc.x(1)

# apply hadamards again
qc.h(0)
qc.h(1)

# measurement
qc.measure([0,1],[0,1])

# draw circuit
print(qc.draw())

# run simulator
simulator = AerSimulator()

job = simulator.run(qc, shots=1000)

result = job.result()
counts = result.get_counts()

print(counts)

     ┌───┐   ┌───┐┌───┐   ┌───┐┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├─■─┤ X ├┤ H ├┤M├───
     ├───┤ │ ├───┤├───┤ │ ├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├─■─┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘   └───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════╩══╩═
                                     0  1 
{'11': 1000}


# 2-Bit Password Search Example

A random 2-bit password is selected from:

00
01
10
11

The oracle acts like a password verification system:
- if the guess is correct, its phase is flipped
- otherwise nothing happens

Grover’s algorithm then amplifies the probability of the correct password using interference.

In [2]:
import random

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# randomly choose 2-bit password
password = random.choice(["00", "01", "10", "11"])

# create circuit
qc = QuantumCircuit(2,2)

# create equal superposition
qc.h(0)
qc.h(1)

# oracle construction

# apply X gates wherever password bit is 0
for i, bit in enumerate(password):
    if bit == "0":
        qc.x(i)

# CZ flips phase only for |11>
qc.cz(0,1)

# undo X gates
for i, bit in enumerate(password):
    if bit == "0":
        qc.x(i)
        
# diffusion operator

# apply Hadamards
qc.h(0)
qc.h(1)

# apply X gates
qc.x(0)
qc.x(1)

# phase flip around |00>
qc.cz(0,1)

# undo X gates
qc.x(0)
qc.x(1)

# apply Hadamards again
qc.h(0)
qc.h(1)

# measurement
qc.measure([0,1],[0,1])

# draw circuit
print(qc.draw())

# run simulator
simulator = AerSimulator()

job = simulator.run(qc, shots=1000)

result = job.result()
counts = result.get_counts()

print("actual password:", password)
print("measurement results:", counts)

     ┌───┐┌───┐   ┌───┐┌───┐┌───┐   ┌───┐┌───┐┌─┐   
q_0: ┤ H ├┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ H ├┤M├───
     ├───┤└───┘ │ ├───┤├───┤└───┘ │ ├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├──────■─┤ H ├┤ X ├──────■─┤ X ├┤ H ├─╫─┤M├
     └───┘        └───┘└───┘        └───┘└───┘ ║ └╥┘
c: 2/══════════════════════════════════════════╩══╩═
                                               0  1 
actual password: 01
measurement results: {'10': 1000}


Initially the measurement output looked reversed.

This happens because Qiskit displays classical bitstrings from right to left.

So a result like:

10

corresponds to:

q_1 = 1
q_0 = 0

which represents the password:

01

# Observation

Initially, all possible states had equal probability.

The oracle then flipped the phase of the correct state without directly changing measurement probabilities.

The diffusion step used interference to redistribute amplitudes:
- amplitudes of incorrect states decreased
- amplitude of the correct state increased

As a result, the probability gradually concentrated onto the marked state.

The final measurement therefore became dominated by the correct password.

# Final Thoughts

At first Grover’s algorithm felt somewhat magical because the oracle only flips phase, yet the final measurement strongly favors the correct answer.

The key idea is that interference continuously reshapes amplitudes:
- wrong answers interfere destructively
- the marked answer interferes constructively

This gradual amplification is what gives Grover its quadratic speedup over classical search.